# Airbnb Rental Price Prediction (Toronto & Vancouver)

**Goal:** Build and evaluate a machine learning model to predict listing rental prices (`price_clean`) based on physical property features and location coordinates.

## Step 1: Data Loading & Stratified Train/Test Split
Load the gold dataset from Delta Lake and split it 80/20 while preserving city proportions across train and test sets.

In [0]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

print("=== Step 1: Loading Gold Feature Table & Train/Test Split ===")

# Read the gold feature dataset from Delta Lake
df_spark = spark.table("airbnb_lakehouse.gold_listings_features")
df = df_spark.toPandas()

print(f"Total dataset loaded: {len(df)} rows across cities: {df['city'].unique().tolist()}")

# Perform 80/20 train/test split stratified per city BEFORE model choice
train_dfs = []
test_dfs = []

for city_name in ["toronto", "vancouver"]:
    city_data = df[df["city"] == city_name].copy()
    train_c, test_c = train_test_split(city_data, test_size=0.20, random_state=42)
    
    train_dfs.append(train_c)
    test_dfs.append(test_c)
    print(f"[{city_name.upper()}] Train set: {len(train_c)} rows | Test set (held-out): {len(test_c)} rows")

df_train = pd.concat(train_dfs, ignore_index=True)
df_test = pd.concat(test_dfs, ignore_index=True)

print(f"\nTOTAL Train set: {len(df_train)} rows | TOTAL Test set: {len(df_test)} rows")

## Step 2: Baseline Model (Median by Room Type & Accommodates)
Build a heuristic benchmark model using median prices for combinations of room type and guest capacity.

In [0]:
print("=== Step 2: Baseline Model (Median by room_type & accommodates) ===")

# Calculate median price on Train set ONLY
baseline_lookup = df_train.groupby(["room_type", "accommodates"])["price_clean"].median().reset_index()
baseline_lookup.rename(columns={"price_clean": "baseline_price"}, inplace=True)

# Fallback global median from train set for unseen combinations
global_train_median = df_train["price_clean"].median()

# Map baseline predictions onto Test set
df_test_eval = df_test.merge(baseline_lookup, on=["room_type", "accommodates"], how="left")
df_test_eval["baseline_price"] = df_test_eval["baseline_price"].fillna(global_train_median)

# Evaluation function for MAE in CAD
def get_mae_report(df_eval, actual_col, pred_col, label="Model"):
    toronto_df = df_eval[df_eval['city'] == 'toronto']
    vancouver_df = df_eval[df_eval['city'] == 'vancouver']
    
    toronto_mae = np.mean(np.abs(toronto_df[actual_col] - toronto_df[pred_col]))
    vancouver_mae = np.mean(np.abs(vancouver_df[actual_col] - vancouver_df[pred_col]))
    total_mae = np.mean(np.abs(df_eval[actual_col] - df_eval[pred_col]))
    
    return {
        "Model": label,
        "Toronto MAE ($)": round(toronto_mae, 2),
        "Vancouver MAE ($)": round(vancouver_mae, 2),
        "Total MAE ($)": round(total_mae, 2)
    }

baseline_report = get_mae_report(df_test_eval, "price_clean", "baseline_price", "Baseline (Median)")
print("Baseline Evaluation Completed.")

## Step 3: ML Model Training (Random Forest Regressor)
Train a Random Forest model with a log transformation on `price_clean` to handle target skewness.

In [0]:
print("=== Step 3: Training ML Price Model (RandomForest) ===")

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# Feature Selection (Intrinsic property physical attributes + location)
cat_features = ["room_type", "property_type", "neighbourhood_cleansed", "host_is_superhost"]
num_features = [
    "accommodates", "bedrooms", "beds", "bathrooms_clean", 
    "latitude", "longitude", "hosts_time_as_host_years", 
    "minimum_nights", "availability_365"
]

# Pipeline Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features)
    ]
)

# ML Model Pipeline
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1))
])

# Prepare Target Variable (Log Transformed)
y_train_log = np.log1p(df_train["price_clean"])

# Enable MLflow Autologging
mlflow.autolog()

with mlflow.start_run(run_name="RandomForest_Price_Model"):
    # 1. Fit model pipeline
    model_pipeline.fit(df_train[num_features + cat_features], y_train_log)
    
    # 2. Predict on Test set and transform back using expm1
    y_test_pred_log = model_pipeline.predict(df_test_eval[num_features + cat_features])
    df_test_eval["predicted_price"] = np.expm1(y_test_pred_log)
    
    # 3. Calculate MAE report
    ml_report = get_mae_report(df_test_eval, "price_clean", "predicted_price", "ML Model (RandomForest)")
    
    # 4. Log custom MAE metrics to MLflow UI
    mlflow.log_metric("toronto_mae", ml_report["Toronto MAE ($)"])
    mlflow.log_metric("vancouver_mae", ml_report["Vancouver MAE ($)"])
    mlflow.log_metric("total_mae", ml_report["Total MAE ($)"])

# Display comparison table
comparison_df = pd.DataFrame([baseline_report, ml_report])
print("\n--- FINAL TEST SET PERFORMANCE COMPARISON ---")
display(comparison_df)

## Step 3.1: Metrics Evaluation & Feature Importance

In [0]:
print("=== Step 3.1: Extended Metrics & Feature Importance Analysis ===")

from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Extended Metrics Function (RMSE, MAE, R2)
def calculate_detailed_metrics(df_eval, actual_col, pred_col, label="Model"):
    metrics = []
    for city_name in ["toronto", "vancouver", "total"]:
        if city_name == "total":
            sub_df = df_eval
        else:
            sub_df = df_eval[df_eval['city'] == city_name]
            
        y_true = sub_df[actual_col]
        y_pred = sub_df[pred_col]
        
        mae = np.mean(np.abs(y_true - y_pred))
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred)
        
        metrics.append({
            "Model": label,
            "Scope": city_name.capitalize(),
            "MAE ($)": round(mae, 2),
            "RMSE ($)": round(rmse, 2),
            "R2 Score": round(r2, 4)
        })
    return pd.DataFrame(metrics)

# Generate detailed metrics for Baseline & ML Model
df_metrics_base = calculate_detailed_metrics(df_test_eval, "price_clean", "baseline_price", "Baseline")
df_metrics_rf = calculate_detailed_metrics(df_test_eval, "price_clean", "predicted_price", "Random Forest")
detailed_summary = pd.concat([df_metrics_base, df_metrics_rf], ignore_index=True)

print("--- EXTENDED METRICS TABLE (TEST SET) ---")
display(detailed_summary)

# 2. Extract and Plot Feature Importances
regressor = model_pipeline.named_steps["regressor"]
preprocessor = model_pipeline.named_steps["preprocessor"]

cat_encoder = preprocessor.named_transformers_["cat"]
encoded_cat_cols = list(cat_encoder.get_feature_names_out(cat_features))
all_feature_names = num_features + encoded_cat_cols

importances = regressor.feature_importances_
feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=False).head(12)

plt.figure(figsize=(10, 5))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette="viridis")
plt.title("Top 12 Most Important Features for Price Prediction", fontsize=13, fontweight="bold")
plt.xlabel("Gini Importance Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

# 3. Scatter Plot: Actual vs Predicted Prices (Test Set)
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_test_eval, x="price_clean", y="predicted_price", hue="city", alpha=0.5, palette="Set1")
plt.plot([0, 1500], [0, 1500], 'r--', label="Perfect Prediction")
plt.xlim(0, 1500)
plt.ylim(0, 1500)
plt.title("Actual vs Predicted Prices (Test Set Truncated at $1500 CAD)", fontsize=13, fontweight="bold")
plt.xlabel("Actual Price (CAD)")
plt.ylabel("Predicted Price (CAD)")
plt.legend()
plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql import functions as F

print("=== Step 4: Exporting All Predictions to Gold Table ===")

# Predict Baseline and ML model on full dataset
df_full = df.merge(baseline_lookup, on=["room_type", "accommodates"], how="left")
df_full["baseline_price"] = df_full["baseline_price"].fillna(global_train_median)

full_pred_log = model_pipeline.predict(df_full[num_features + cat_features])
df_full["predicted_price"] = np.expm1(full_pred_log)

# Round numeric values for clean output
df_full["predicted_price"] = df_full["predicted_price"].round(2)
df_full["baseline_price"] = df_full["baseline_price"].round(2)

# Convert to Spark DataFrame
gold_pred_df = spark.createDataFrame(df_full[[
    "id", "city", "room_type", "accommodates", "price_clean", "baseline_price", "predicted_price"
]])

# Add calculated Absolute Error column
gold_pred_df = gold_pred_df.withColumnRenamed("price_clean", "actual_price") \
                           .withColumn("absolute_error", F.round(F.abs(F.col("actual_price") - F.col("predicted_price")), 2))

# Write to Partitioned Gold Table
gold_pred_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("airbnb_lakehouse.gold_model_predictions")

print("Successfully saved full model predictions to table: 'airbnb_lakehouse.gold_model_predictions'")

## Results Interpretation & Conclusions

### 1. Model Performance
- **Random Forest** achieved a significant improvement over the Baseline model: overall MAE dropped from **$109.32** down to **$68.53** (a **37%** increase in accuracy).
- The coefficient of determination (**R² = 0.626**) indicates that the model accounts for over 62% of price variance.

### 2. City-by-City Comparison (Toronto vs. Vancouver)
- **Toronto (MAE: $61.24 | R²: 0.6263):** Yields higher accuracy due to a larger training sample (13,380 rows).
- **Vancouver (MAE: $92.25 | R²: 0.5692):** Higher average error stems from fewer training instances (4,116 rows) and greater price dispersion in the Vancouver market.

### 3. Key Price Drivers (Feature Importance)
Based on feature importance scoring:
1. **Location (`latitude`, `longitude`):** Geographic coordinates represent the single most important price predictor.
2. **Capacity & Comfort (`accommodates`, `bathrooms_clean`, `bedrooms`):** Guest allowance and bathroom count dictate the base tier of the listing.
3. **Property Category (`room_type`):** Offering an entire place versus a private room creates a major price step-up.

### 4. Next Steps & Recommendations
- Feature engineering: Incorporate text features (NLP on listing descriptions) or seasonal trend indicators.
- Model exploration: Test gradient boosting methods (XGBoost / LightGBM), which often perform exceptionally well on tabular data.